# Heart Signal Reconstruction & Separate LMS Hyperparameter Optimization Pipeline

This notebook presents a clean, streamlined end-to-end pipeline for **reconstructing the cardiac signal** and **estimating Heart Rate (BPM)** from individual 5-dimensional wrist-worn PPG and triaxial accelerometer windows (`[ppg1, ppg2, accx, accy, accz]`, $N = 1000$ samples at $f_s = 125\text{ Hz}$, i.e., $8\text{ s}$ per window).

---

## 1. Pipeline Architecture

To make the **Adaptive MISO LMS filter** the true centerpiece of motion artifact removal, preprocessing and BPM detection are restricted to their essential minimum (no artificial spectral subtraction or ad-hoc post-filtering):

1. **Minimal Preprocessing (`preprocessing.py`)**:
   - **Zero-Phase Bandpass Filter (`0.5 Hz – 4.0 Hz` $\equiv$ `30 BPM – 240 BPM`)**: 4th-order Butterworth `sosfiltfilt` applied to the 5 channels `[ppg1, ppg2, accx, accy, accz]`.
   - **Amplitude Normalization**: Each channel is normalized to zero mean and unit standard deviation.

2. **MISO Reference Signal Construction (`reference_construction.py`)**:
   - Constructs the $3M$-tap spatio-temporal regressor vector $\mathbf{a}[n] = [\mathbf{a}_1^T[n], \mathbf{a}_2^T[n], \mathbf{a}_3^T[n]]^T \in \mathbb{R}^{3M}$ from the 3 normalized accelerometer axes (`accx, accy, accz`):
     $$\hat{m}[n] = \sum_{k=1}^{3} \mathbf{w}_k^T[n]\mathbf{a}_k[n] = \mathbf{w}^T[n]\mathbf{a}[n]$$

3. **Parallel Adaptive MISO LMS Filtering & SNR-Weighted Late Fusion (`LMS_algorithms.py` & `fusion.py`)**:
   - Filters `ppg1` ($d_1[n]$) and `ppg2` ($d_2[n]$) in parallel against $\mathbf{a}[n]$ to obtain the cleared cardiac signals $e_1[n] = d_1[n] - \hat{m}_1[n]$ and $e_2[n] = d_2[n] - \hat{m}_2[n]$.
   - Evaluates the **3 LMS variants separately**:
     - **Standard LMS (`standard_lms`)**: $\mathbf{w}[n+1] = \mathbf{w}[n] + \mu e[n]\mathbf{a}[n]$ (Hyperparameters: $M, \mu$)
     - **Normalized LMS (`nlms`)**: $\mathbf{w}[n+1] = \mathbf{w}[n] + \frac{\mu}{\delta + \|\mathbf{a}[n]\|^2} e[n]\mathbf{a}[n]$ (Hyperparameters: $M, \mu, \delta$)
     - **Variable Step-Size LMS (`vs_lms`)**: $\mathbf{w}[n+1] = \mathbf{w}[n] + \frac{\mu[n]}{\delta + \|\mathbf{a}[n]\|^2} e[n]\mathbf{a}[n]$, $\mu[n+1] = \text{clip}(\alpha \mu[n] + \gamma e^2[n], \mu_{\min}, \mu_{\max})$ (Hyperparameters: $M, \mu_{\max}, \mu_{\min}, \alpha, \gamma$)
   - **SNR-Weighted Late Fusion**: Combines $e_1[n]$ and $e_2[n]$ using their in-band cardiac spectral SNR weights $w_i = \frac{\text{SNR}_i}{\text{SNR}_1 + \text{SNR}_2}$:
     $$e_{\text{fused}}[n] = w_1 e_1[n] + w_2 e_2[n]$$

4. **Direct Heart Rate Detection (`bpm_detection.py`)**:
   - **Zero-Padded Periodogram (`periodogram`)**: Hann/Hamming/Blackman tapered, zero-padded FFT peak detection in `[0.5, 4.0] Hz`.
   - **Elgendi's Systolic Peak Detection (`elgendi`)**: Two-moving-average systolic peak detector.

5. **Leakage-Free Subject-Wise Group K-Fold Optimization (`optimization.py`)**:
   - Uses `GroupKFold` partitioned strictly by recording session / subject ID so overlapping windows from the same person never leak between training and validation splits.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "restructured_implementation":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath(".")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from restructured_implementation import (
    FS,
    WIN,
    load_ts,
    get_participant_groups,
    preprocess_signal,
    snr_weighted_late_fusion,
    zero_padded_periodogram_bpm,
    objective_function,
    optimize_hr_pipeline,
)

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

train_path = os.path.join(PROJECT_ROOT, "IEEEPPG_TRAIN.ts")
test_path = os.path.join(PROJECT_ROOT, "IEEEPPG_TEST.ts")

X_train, y_train = load_ts(train_path)
X_test, y_test = load_ts(test_path)

train_groups = get_participant_groups(X_train)
test_groups = get_participant_groups(X_test)

print(f"Training Set : X_train = {X_train.shape}, y_train = {y_train.shape} | Subjects = {len(np.unique(train_groups))}")
print(f"Test Set     : X_test  = {X_test.shape},  y_test  = {y_test.shape}  | Subjects = {len(np.unique(test_groups))}")

---
## 2. Separate Hyperparameter Optimization of the 3 LMS Variants (Group K-Fold CV)

We optimize `standard_lms`, `nlms`, and `vs_lms` **separately** using 5-fold Subject-Wise Group K-Fold Cross-Validation and compare each variant against the **Before-LMS Baseline** (preprocessed PPG without LMS adaptive filtering).

In [ ]:
opt_results = optimize_hr_pipeline(
    X_train,
    y_train,
    n_splits=5,
    phase1_trials=36,
    phase1_sample_size=150,
    phase1_top_k=6,
    phase2_trials=42,
    prune_percentile=50.0,
    seed=42,
)

best_model = opt_results["best_model"]
best_hyperparams = opt_results["best_hyperparameters"]
results_by_lms = opt_results["results_by_lms"]
baseline_cv = opt_results["baseline_cv"]

print("=" * 92)
print("SELECTED HYPERPARAMETERS & 5-FOLD SUBJECT-WISE CV PERFORMANCE BY LMS VARIANT")
print("=" * 92)
print(
    f"{'Before LMS (Baseline)':<22} | {'Obj':>7}: {baseline_cv['score_objective']:6.2f} | "
    f"{'AAE':>6}: {baseline_cv['score_aae']:6.2f} BPM | "
    f"{'RMSE':>6}: {baseline_cv['score_rmse']:6.2f} BPM | "
    f"Outliers (>10 BPM): {baseline_cv['outlier_pct']:5.1f}%"
)
print("-" * 92)

for lms_name, info in results_by_lms.items():
    hp = info["hyperparameters"]
    lms_p_str = ", ".join(
        f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}"
        for k, v in hp["lms_params"].items()
    )
    hr_p_str = ", ".join(
        f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}"
        for k, v in hp["hr_params"].items()
    )
    print(
        f"{lms_name:<22} | {'Obj':>7}: {info['score_objective']:6.2f} | "
        f"{'AAE':>6}: {info['score_aae']:6.2f} BPM | "
        f"{'RMSE':>6}: {info['score_rmse']:6.2f} BPM | "
        f"Outliers (>10 BPM): {info['outlier_pct']:5.1f}%"
    )
    print(f"   -> LMS Params : {lms_p_str}")
    print(f"   -> HR Detector: {hp['hr_algorithm']} ({hr_p_str})")
    print("-" * 92)

print(f"OVERALL WINNING PIPELINE: {best_hyperparams['lms_algorithm']} + {best_hyperparams['hr_algorithm']}")
print(f"   -> Winning LMS Hyperparameters: {best_hyperparams['lms_params']}")
print(f"   -> Winning HR Hyperparameters : {best_hyperparams['hr_params']}")
print("=" * 92)

---
## 3. Test Set Evaluation (`IEEEPPG_TEST.ts`) & Spread of Test BPMs Colored by Subject

We now apply each of the 3 optimized LMS models (`standard_lms`, `nlms`, `vs_lms`) as well as the unfiltered baseline (`Before LMS`) to the held-out test set `IEEEPPG_TEST.ts`, treating every 1000-sample window independently:
- Output the Test Set performance metrics for all 3 LMS types and the overall winner.
- Show the **spread of Test BPMs (Estimated vs. Ground-Truth)** colored by subject.
- Show the **Test Set performance broken down by subject**.

In [ ]:
# 1. Compute Before-LMS Baseline on the Test Set
ppg1_test, ppg2_test, acc_test = preprocess_signal(X_test)
y_pred_before_lms = np.array([
    zero_padded_periodogram_bpm(snr_weighted_late_fusion(ppg1_test[i], ppg2_test[i]), nfft=8192)
    for i in range(len(X_test))
])

def compute_metrics(y_hat, y_true):
    abs_e = np.abs(y_hat - y_true)
    return {
        "obj": objective_function(y_hat, y_true),
        "aae": float(np.mean(abs_e)),
        "rmse": float(np.sqrt(np.mean(abs_e ** 2))),
        "medae": float(np.median(abs_e)),
        "outlier_pct": float(np.mean(abs_e > 10.0) * 100.0),
        "abs_errors": abs_e,
    }

test_eval = {"Before LMS": {"preds": y_pred_before_lms, **compute_metrics(y_pred_before_lms, y_test)}}
cleaned_signals_by_lms = {}

for lms_name, info in results_by_lms.items():
    preds, sigs = info["model"].predict_with_signals(X_test)
    cleaned_signals_by_lms[lms_name] = sigs
    test_eval[lms_name] = {"preds": preds, **compute_metrics(preds, y_test)}

win_name = best_hyperparams["lms_algorithm"]
y_test_pred = test_eval[win_name]["preds"]
cleaned_test_signals = cleaned_signals_by_lms[win_name]
abs_errors = test_eval[win_name]["abs_errors"]
outlier_mask = abs_errors > 10.0

print("=" * 92)
print("TEST SET PERFORMANCE COMPARISON (IEEEPPG_TEST.ts — BEFORE LMS vs. 3 LMS VARIANTS)")
print("=" * 92)
print(f"{'Method':<18} | {'Objective':>10} | {'AAE (BPM)':>10} | {'RMSE (BPM)':>10} | {'MedAE (BPM)':>11} | {'Outliers (>10 BPM)':>18}")
print("-" * 92)
for name, m in test_eval.items():
    marker = " *WINNER*" if name == win_name else ""
    print(
        f"{name:<18} | {m['obj']:10.2f} | {m['aae']:10.2f} | "
        f"{m['rmse']:10.2f} | {m['medae']:11.2f} | {m['outlier_pct']:17.2f}%{marker}"
    )
print("=" * 92)

# Per-subject breakdown for the winning model
unique_test_subjects = np.unique(test_groups)
subj_aae, subj_obj, subj_out = [], [], []
print("\nTEST SET PERFORMANCE BY SUBJECT (WINNING LMS PIPELINE):")
print(f"{'Subject':<12} | {'Windows':>8} | {'AAE (BPM)':>10} | {'Objective':>10} | {'Outliers (>10 BPM)':>18}")
print("-" * 68)
for s in unique_test_subjects:
    idx = test_groups == s
    m_s = compute_metrics(y_test_pred[idx], y_test[idx])
    subj_aae.append(m_s["aae"])
    subj_obj.append(m_s["obj"])
    subj_out.append(m_s["outlier_pct"])
    print(f"Subject {s + 1:2d}   | {int(np.sum(idx)):8d} | {m_s['aae']:10.2f} | {m_s['obj']:10.2f} | {m_s['outlier_pct']:17.1f}%")

# Plot 1: Spread of Test BPMs Colored by Subject + Method Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cmap = plt.cm.get_cmap("tab10", len(unique_test_subjects))
for idx_s, s in enumerate(unique_test_subjects):
    mask = test_groups == s
    axes[0].scatter(
        y_test[mask],
        y_test_pred[mask],
        color=cmap(idx_s),
        alpha=0.75,
        s=28,
        edgecolors="none",
        label=f"Subj {s + 1} (AAE={subj_aae[idx_s]:.1f})",
    )

lims = [min(y_test.min(), y_test_pred.min()) - 5, max(y_test.max(), y_test_pred.max()) + 5]
axes[0].plot(lims, lims, "k--", lw=1.5, label="Ideal (y = x)")
axes[0].plot(lims, [l + 10 for l in lims], "r:", lw=1.0, label="+/- 10 BPM Outlier Bound")
axes[0].plot(lims, [l - 10 for l in lims], "r:", lw=1.0)
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_xlabel("Ground-Truth Heart Rate (BPM)")
axes[0].set_ylabel("Estimated Heart Rate (BPM)")
axes[0].set_title(f"Spread of Test Set BPMs Colored by Subject ({win_name})")
axes[0].legend(loc="upper left", fontsize=8, ncol=2)

# Right subplot: Before LMS vs 3 LMS Variants on Test Set & Per-Subject AAE
methods = list(test_eval.keys())
aaes = [test_eval[k]["aae"] for k in methods]
objs = [test_eval[k]["obj"] for k in methods]
x_pos = np.arange(len(methods))
bar_colors = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]

bars1 = axes[1].bar(x_pos - 0.18, aaes, width=0.35, color=bar_colors, alpha=0.9, label="Test AAE (BPM)")
bars2 = axes[1].bar(x_pos + 0.18, objs, width=0.35, color=bar_colors, alpha=0.45, edgecolor="black", label="Test Objective")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel("Error / Score (Lower is Better)")
axes[1].set_title("Test Set BPM Improvement: Before LMS vs. 3 LMS Variants")
axes[1].legend()
for b in bars1:
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, f"{b.get_height():.2f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 4. Visual Progression of a Single Window Over Each Step of the Pipeline

We select a representative motion-corrupted window from the test set and visualize how the signal transforms across each stage of the pipeline:
1. **Raw 5D Sensor Input**: Dual PPG channels (`ppg1`, `ppg2`) and 3-axis accelerometer (`accx`, `accy`, `accz`).
2. **Minimal Preprocessing**: Bandpass filtered (`0.5–4.0 Hz`) and normalized to zero mean and unit variance.
3. **Parallel MISO LMS Filtering**: Primary input $d_1[n]$, estimated motion artifact $\hat{m}_1[n] = \mathbf{w}_1^T[n]\mathbf{a}[n]$, and cleared cardiac error output $e_1[n] = d_1[n] - \hat{m}_1[n]$.
4. **Before vs. After LMS Recombined Signal & Spectrum**: Pre-LMS fused signal $d_{\text{fused}}[n]$ vs. post-LMS fused signal $e_{\text{fused}}[n]$ and their zero-padded periodograms relative to the Ground-Truth BPM.

In [ ]:
# Pick a window where LMS substantially improves BPM estimation over the unfiltered baseline
before_err = np.abs(y_pred_before_lms - y_test)
after_err = np.abs(y_test_pred - y_test)
improvement = before_err - after_err
demo_idx = int(np.argmax(improvement))
if improvement[demo_idx] <= 0:
    demo_idx = int(np.argmin(after_err))

demo_win = X_test[demo_idx]
demo_true_bpm = float(y_test[demo_idx])
e_Demo, demo_details = best_model.reconstruct_window(demo_win, return_details=True)
d_Demo = demo_details["d_fused"]
demo_before_bpm = float(y_pred_before_lms[demo_idx])
demo_after_bpm = float(y_test_pred[demo_idx])

t_axis = np.arange(WIN) / FS

fig, axes = plt.subplots(4, 2, figsize=(15, 13))

# Row 1: Raw PPG & Raw Accelerometer
axes[0, 0].plot(t_axis, demo_win[0], label="Raw PPG 1", color="#1f77b4", lw=1.2)
axes[0, 0].plot(t_axis, demo_win[1], label="Raw PPG 2", color="#ff7f0e", lw=1.2, alpha=0.85)
axes[0, 0].set_title(f"Step 1a: Raw Dual PPG Input (Window #{demo_idx})")
axes[0, 0].set_ylabel("Raw Amplitude")
axes[0, 0].legend(loc="upper right", fontsize=8)

axes[0, 1].plot(t_axis, demo_win[2], label="Raw Acc X", lw=1.1)
axes[0, 1].plot(t_axis, demo_win[3], label="Raw Acc Y", lw=1.1)
axes[0, 1].plot(t_axis, demo_win[4], label="Raw Acc Z", lw=1.1)
axes[0, 1].set_title("Step 1b: Raw 3-Axis Accelerometer Reference")
axes[0, 1].set_ylabel("Raw Acceleration")
axes[0, 1].legend(loc="upper right", fontsize=8)

# Row 2: Preprocessed (Bandpass 0.5-4.0 Hz + Normalized)
axes[1, 0].plot(t_axis, demo_details["ppg1"], label="Preprocessed d1[n]", color="#1f77b4", lw=1.2)
axes[1, 0].plot(t_axis, demo_details["ppg2"], label="Preprocessed d2[n]", color="#ff7f0e", lw=1.2, alpha=0.85)
axes[1, 0].set_title("Step 2a: Bandpass (0.5–4.0 Hz) + Amplitude-Normalized PPG")
axes[1, 0].set_ylabel("Normalized Amp")
axes[1, 0].legend(loc="upper right", fontsize=8)

axes[1, 1].plot(t_axis, demo_details["acc"][0], label="Norm Acc X", lw=1.1)
axes[1, 1].plot(t_axis, demo_details["acc"][1], label="Norm Acc Y", lw=1.1)
axes[1, 1].plot(t_axis, demo_details["acc"][2], label="Norm Acc Z", lw=1.1)
axes[1, 1].set_title("Step 2b: Bandpass (0.5–4.0 Hz) + Normalized 3-Axis Accelerometer")
axes[1, 1].set_ylabel("Normalized Amp")
axes[1, 1].legend(loc="upper right", fontsize=8)

# Row 3: Adaptive MISO LMS Filtering on Branch 1 & Branch 2
axes[2, 0].plot(t_axis, demo_details["ppg1"], label="Primary Input d1[n]", color="gray", alpha=0.5, lw=1.0)
axes[2, 0].plot(t_axis, demo_details["m_hat1"], label="LMS Motion Estimate m_hat1[n]", color="#d62728", lw=1.2, ls="--")
axes[2, 0].plot(t_axis, demo_details["e1"], label="Cleaned Cardiac e1[n]", color="#2ca02c", lw=1.5)
axes[2, 0].set_title(f"Step 3a: Branch 1 MISO {best_model.lms_algorithm.upper()} (M={best_model.M})")
axes[2, 0].set_ylabel("Amplitude")
axes[2, 0].legend(loc="upper right", fontsize=8)

f_info = demo_details["fusion_info"]
axes[2, 1].plot(t_axis, demo_details["e1"], label=f"e1[n] (SNR={f_info['snr1']:.2f}, w1={f_info['w1']:.2f})", color="#1f77b4", alpha=0.65, lw=1.1)
axes[2, 1].plot(t_axis, demo_details["e2"], label=f"e2[n] (SNR={f_info['snr2']:.2f}, w2={f_info['w2']:.2f})", color="#ff7f0e", alpha=0.65, lw=1.1)
axes[2, 1].plot(t_axis, e_Demo, label="Late-Fused Cardiac Signal e_fused[n]", color="#2ca02c", lw=1.8)
axes[2, 1].set_title("Step 3b: SNR-Weighted Late Fusion of Parallel LMS Branches")
axes[2, 1].set_ylabel("Amplitude")
axes[2, 1].legend(loc="upper right", fontsize=8)

# Row 4: Recombined Signal Before vs. After LMS (Time Domain & Periodogram Spectrum)
axes[3, 0].plot(t_axis, d_Demo, label=f"Before LMS d_fused[n] (Est={demo_before_bpm:.1f} BPM)", color="#d62728", alpha=0.7, lw=1.2)
axes[3, 0].plot(t_axis, e_Demo, label=f"After LMS e_fused[n] (Est={demo_after_bpm:.1f} BPM)", color="#2ca02c", lw=1.6)
axes[3, 0].set_title(f"Step 4a: Recombined Signal Before vs. After LMS (True = {demo_true_bpm:.1f} BPM)")
axes[3, 0].set_xlabel("Time (s)")
axes[3, 0].set_ylabel("Amplitude")
axes[3, 0].legend(loc="upper right", fontsize=8)

nfft_plot = 8192
freqs_bpm = 60.0 * np.fft.rfftfreq(nfft_plot, d=1.0 / FS)
band_m = (freqs_bpm >= 30.0) & (freqs_bpm <= 240.0)
spec_before = np.abs(np.fft.rfft((d_Demo - d_Demo.mean()) * np.hanning(WIN), n=nfft_plot))[band_m]
spec_after = np.abs(np.fft.rfft((e_Demo - e_Demo.mean()) * np.hanning(WIN), n=nfft_plot))[band_m]
spec_before /= (spec_before.max() + 1e-12)
spec_after /= (spec_after.max() + 1e-12)

axes[3, 1].plot(freqs_bpm[band_m], spec_before, label=f"Before LMS Spectrum (Peak={demo_before_bpm:.1f} BPM)", color="#d62728", alpha=0.7, lw=1.3)
axes[3, 1].plot(freqs_bpm[band_m], spec_after, label=f"After LMS Spectrum (Peak={demo_after_bpm:.1f} BPM)", color="#2ca02c", lw=1.7)
axes[3, 1].axvline(demo_true_bpm, color="black", ls="--", lw=1.5, label=f"Ground-Truth ({demo_true_bpm:.1f} BPM)")
axes[3, 1].set_title("Step 4b: Zero-Padded Periodogram Before vs. After LMS")
axes[3, 1].set_xlabel("Heart Rate Frequency (BPM)")
axes[3, 1].set_ylabel("Normalized Magnitude")
axes[3, 1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

---
## 5. Outlier vs. Good Example: Raw Signal Above & LMS-Cleaned Signal Below

Finally, we compute the fraction of test windows that are outliers ($|\hat{y} - y_{\text{true}}| > 10\text{ BPM}$) and compare:
- **Left Column (Good Example)**: Raw preprocessed PPG signal on top and its LMS-cleaned cardiac waveform directly below it.
- **Right Column (Outlier Example)**: Raw preprocessed PPG signal of a severe motion-corrupted outlier on top and its LMS-cleaned waveform directly below it.

In [ ]:
n_outliers = int(np.sum(outlier_mask))
outlier_fraction = float(np.mean(outlier_mask))
print(f"Test Set Outlier Fraction (|Error| > 10 BPM): {n_outliers} / {len(y_test)} windows ({outlier_fraction * 100:.2f}%)")
print(f"Well-Detected Fraction    (|Error| <= 10 BPM): {len(y_test) - n_outliers} / {len(y_test)} windows ({(1.0 - outlier_fraction) * 100:.2f}%)")

good_idx = int(np.argmin(abs_errors))
outlier_idx = int(np.argmax(abs_errors))

e_good, good_det = best_model.reconstruct_window(X_test[good_idx], return_details=True)
e_out, out_det = best_model.reconstruct_window(X_test[outlier_idx], return_details=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)

# Left Column: Good Example (Top = Raw Preprocessed Signal, Bottom = Cleaned Signal)
axes[0, 0].plot(t_axis, good_det["ppg1"], label="Raw Preprocessed PPG1", color="#1f77b4", lw=1.2)
axes[0, 0].plot(t_axis, good_det["ppg2"], label="Raw Preprocessed PPG2", color="#ff7f0e", lw=1.2, alpha=0.8)
axes[0, 0].set_title(
    f"GOOD EXAMPLE (Window #{good_idx}) — Raw Signal Before LMS\n"
    f"True BPM = {y_test[good_idx]:.1f} BPM | Before-LMS Est = {y_pred_before_lms[good_idx]:.1f} BPM"
)
axes[0, 0].set_ylabel("Normalized Amplitude")
axes[0, 0].legend(loc="upper right", fontsize=8)

axes[1, 0].plot(t_axis, e_good, label=f"LMS-Cleaned Signal e_fused[n]", color="#2ca02c", lw=1.6)
axes[1, 0].set_title(
    f"GOOD EXAMPLE (Window #{good_idx}) — Cleaned Signal After LMS\n"
    f"True BPM = {y_test[good_idx]:.1f} BPM | After-LMS Est = {y_test_pred[good_idx]:.1f} BPM (Error = {abs_errors[good_idx]:.2f} BPM)"
)
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].set_ylabel("Normalized Amplitude")
axes[1, 0].legend(loc="upper right", fontsize=8)

# Right Column: Outlier Example (Top = Raw Preprocessed Signal, Bottom = Cleaned Signal)
axes[0, 1].plot(t_axis, out_det["ppg1"], label="Raw Preprocessed PPG1", color="#1f77b4", lw=1.2)
axes[0, 1].plot(t_axis, out_det["ppg2"], label="Raw Preprocessed PPG2", color="#ff7f0e", lw=1.2, alpha=0.8)
axes[0, 1].set_title(
    f"OUTLIER EXAMPLE (Window #{outlier_idx}) — Raw Signal Before LMS\n"
    f"True BPM = {y_test[outlier_idx]:.1f} BPM | Before-LMS Est = {y_pred_before_lms[outlier_idx]:.1f} BPM"
)
axes[0, 1].set_ylabel("Normalized Amplitude")
axes[0, 1].legend(loc="upper right", fontsize=8)

axes[1, 1].plot(t_axis, e_out, label="LMS-Cleaned Signal e_fused[n]", color="#d62728", lw=1.6)
axes[1, 1].set_title(
    f"OUTLIER EXAMPLE (Window #{outlier_idx}) — Cleaned Signal After LMS\n"
    f"True BPM = {y_test[outlier_idx]:.1f} BPM | After-LMS Est = {y_test_pred[outlier_idx]:.1f} BPM (Error = {abs_errors[outlier_idx]:.2f} BPM)"
)
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Normalized Amplitude")
axes[1, 1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()